# EDA: โอกาสปล่อยเช่าภายใน 4 สัปดาห์

Notebook นี้สำรวจปัจจัยที่ทราบแล้วในวันที่ลงประกาศ เพื่อเตรียมข้อมูลสำหรับ Logistic Regression และ CatBoost โดยไม่ใช้ข้อมูลที่เกิดหลังจากปล่อยเช่า

## 1. Import และกำหนดตำแหน่งโปรเจกต์

Cell นี้ทำให้เปิด Notebook ได้ทั้งจากโฟลเดอร์หลักและจากโฟลเดอร์ `notebooks/`

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.calibration import calibration_curve

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from kaverentai.data.load_data import load_all_data
from kaverentai.features.lease_probability_features import (
    MODEL_FEATURES,
    TARGET_COLUMN,
    build_lease_probability_data,
)

sns.set_theme(style="whitegrid")

## 2. สร้าง Modeling Dataset

Target คือ `leased_within_4_weeks` ซึ่งตอบคำถามว่า ประกาศนี้ปล่อยเช่าได้ภายใน 4 สัปดาห์หรือไม่ รายการช่วงท้ายที่มีเวลาติดตามไม่ครบ 4 สัปดาห์จะไม่ถูกนำมาใช้

รายการที่ลงท้ายด้วย `-B` เป็นข้อมูลทางเลือกของประกาศเดิม โดยใช้ห้อง วันลงประกาศ และผลลัพธ์เดียวกับแถวหลัก จึงเก็บไว้ในข้อมูลต้นฉบับเพื่อ audit แต่ไม่นับซ้ำใน Modeling Dataset

In [ ]:
tables = load_all_data(cleaned=True)
lease_data = build_lease_probability_data(
    listings=tables["listings"],
    weekly_market=tables["weekly_market"],
)

print(f"จำนวนแถว: {len(lease_data):,}")
print(f"อัตราปล่อยเช่าภายใน 4 สัปดาห์: {lease_data[TARGET_COLUMN].mean():.1%}")
lease_data.head()

## 3. ตรวจ Data Leakage

ตัวแปรอย่าง `weeks_on_market`, `n_viewings`, `week_leased`, `tenant_segment` และ `lease_months` เกิดหลังวันลงประกาศ จึงห้ามใช้เป็น Feature

In [ ]:
post_listing_columns = {
    "asking_rent", "weeks_on_market", "n_viewings",
    "week_leased", "tenant_segment", "lease_months", "leased",
}

assert post_listing_columns.isdisjoint(MODEL_FEATURES)
print("ผ่าน: Feature ไม่มีข้อมูลที่เกิดหลังวันลงประกาศ")
print(f"จำนวน Feature: {len(MODEL_FEATURES)}")

## 4. แบ่งข้อมูลตามเวลา

ใช้ Train สำหรับ EDA และเลือก Feature ส่วน Validation ใช้เปรียบเทียบโมเดล และ Test เก็บไว้ประเมินครั้งสุดท้าย

In [ ]:
split_summary = (
    lease_data.groupby("data_split")[TARGET_COLUMN]
    .agg(rows="size", leased_within_4_weeks="sum", positive_rate="mean")
)
split_summary.style.format({"positive_rate": "{:.1%}"})

In [ ]:
train_data = lease_data.loc[lease_data["data_split"].eq("train")].copy()

plt.figure(figsize=(6, 4))
sns.countplot(data=train_data, x=TARGET_COLUMN, color="#457b9d")
plt.title("Target Distribution: Train Only")
plt.xlabel("Leased within 4 weeks")
plt.ylabel("Listings")
plt.show()

## 5. ความสัมพันธ์กับราคาและสภาวะตลาด

`rent_to_market_ratio` มากกว่า 1 หมายถึงราคาเริ่มต้นสูงกว่าค่ากลางของตลาดในโครงการและสัปดาห์เดียวกัน

In [ ]:
plot_data = train_data.assign(
    target_label=train_data[TARGET_COLUMN].map(
        {True: "Leased within 4 weeks", False: "Not within 4 weeks"}
    )
)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.boxplot(
    data=plot_data, x="target_label", y="rent_to_market_ratio",
    showfliers=False, ax=axes[0],
)
sns.boxplot(
    data=plot_data, x="target_label", y="market_demand_pressure",
    showfliers=False, ax=axes[1],
)
axes[0].set_title("Starting Rent Compared with Market")
axes[1].set_title("Market Demand Pressure")
for axis in axes:
    axis.tick_params(axis="x", rotation=10)
plt.tight_layout()
plt.show()

## 6. เปรียบเทียบตามฤดูกาลและโครงการ

In [ ]:
season_summary = (
    train_data.groupby("season_listed", as_index=False)
    .agg(listings=(TARGET_COLUMN, "size"), positive_rate=(TARGET_COLUMN, "mean"))
    .sort_values("positive_rate", ascending=False)
)
season_summary.style.format({"positive_rate": "{:.1%}"})

In [ ]:
project_summary = (
    train_data.groupby("project_id", as_index=False)
    .agg(listings=(TARGET_COLUMN, "size"), positive_rate=(TARGET_COLUMN, "mean"))
    .sort_values("positive_rate", ascending=False)
)

plt.figure(figsize=(10, 5))
sns.barplot(data=project_summary, x="project_id", y="positive_rate", color="#2a9d8f")
plt.title("Four-week Lease Rate by Project: Train Only")
plt.ylabel("Positive rate")
plt.show()

## 7. สรุป

- ใช้กรอบเวลา 4 สัปดาห์เพื่อให้ Target มีความหมายทางธุรกิจและแก้ปัญหา right-censoring
- ใช้เฉพาะข้อมูลที่ทราบในวันลงประกาศ
- ราคาเทียบตลาด สภาวะอุปสงค์ ฤดูกาล และโครงการเป็นกลุ่ม Feature สำคัญที่ควรนำไปทดสอบ
- ความแตกต่างของ positive rate ตามช่วงเวลาแสดงว่าต้องประเมินด้วย chronological split เท่านั้น

## 8. เอกสารอ้างอิง

- Prokhorenkova, L., Gusev, G., Vorobev, A., Dorogush, A. V., & Gulin, A. (2018). *CatBoost: Unbiased boosting with categorical features*. Advances in Neural Information Processing Systems 31. https://proceedings.neurips.cc/paper/2018/hash/14491b756b3a51daac41c24863285549-Abstract.html
- Pedregosa, F., et al. (2011). Scikit-learn: Machine learning in Python. *Journal of Machine Learning Research, 12*, 2825–2830. https://www.jmlr.org/papers/volume12/pedregosa11a/pedregosa11a.pdf
- Heagerty, P. J., Lumley, T., & Pepe, M. S. (2000). Time-dependent ROC curves for censored survival data and a diagnostic marker. *Biometrics, 56*(2), 337–344. https://doi.org/10.1111/j.0006-341X.2000.00337.x
- Scikit-learn developers. *TimeSeriesSplit*. https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.TimeSeriesSplit.html
- Scikit-learn developers. *Metrics and scoring*. https://scikit-learn.org/stable/modules/model_evaluation.html

คำอธิบายภาษาไทยและตัวอย่างข้อความสำหรับใส่รายงานอยู่ที่ `docs/model_references.md`

# ส่วนที่ 2: Model Interpretation และ Error Analysis

ส่วนนี้อ่านผลลัพธ์ที่สร้างจาก `train_lease_probability.py` เพื่ออธิบายว่าโมเดลตัดสินใจอย่างไร Probability น่าเชื่อถือเพียงใด และโมเดลพลาดในกรณีใด

In [ ]:
metrics_path = PROJECT_ROOT / "reports" / "metrics" / "lease_probability_metrics.json"
tables_dir = PROJECT_ROOT / "reports" / "tables"

with metrics_path.open(encoding="utf-8") as metrics_file:
    model_metrics = json.load(metrics_file)

validation_predictions = pd.read_csv(
    tables_dir / "lease_probability_validation_predictions.csv",
    parse_dates=["date_listed"],
)
test_predictions = pd.read_csv(
    tables_dir / "lease_probability_test_predictions.csv",
    parse_dates=["date_listed"],
)
feature_importance = pd.read_csv(
    tables_dir / "lease_probability_feature_importance.csv"
)
threshold_analysis = pd.read_csv(
    tables_dir / "lease_probability_threshold_analysis.csv"
)
feature_ablation = pd.read_csv(
    tables_dir / "lease_probability_feature_ablation.csv"
)
rolling_metrics = pd.read_csv(
    tables_dir / "lease_probability_rolling_metrics.csv"
)

selected_model = model_metrics["model_settings"]["selected_model"]
test_metrics = model_metrics["test"][selected_model]
print("Selected model:", selected_model)

## 9. Confusion Matrix

Confusion Matrix ช่วยแยกข้อผิดพลาดสองแบบ: False Positive คือคาดว่าจะปล่อยได้เร็วแต่จริงไม่ทัน 4 สัปดาห์ ส่วน False Negative คือคาดว่าปล่อยช้าแต่จริงปล่อยได้เร็ว

In [ ]:
confusion = np.array([
    [test_metrics["true_negative"], test_metrics["false_positive"]],
    [test_metrics["false_negative"], test_metrics["true_positive"]],
])

plt.figure(figsize=(6, 5))
sns.heatmap(
    confusion, annot=True, fmt=",d", cmap="Blues", cbar=False,
    xticklabels=["Predicted slow", "Predicted within 4 weeks"],
    yticklabels=["Actual slow", "Actual within 4 weeks"],
)
plt.title("Test Confusion Matrix at Threshold 0.50")
plt.xlabel("Prediction")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

pd.Series({
    "Precision": test_metrics["precision"],
    "Recall / Sensitivity": test_metrics["recall"],
    "Specificity": test_metrics["specificity"],
    "Balanced Accuracy": test_metrics["balanced_accuracy"],
}).to_frame("Test value").style.format("{:.3f}")

## 10. Probability Calibration

Calibration Curve เปรียบเทียบ Probability เฉลี่ยของโมเดลกับสัดส่วนที่เกิดขึ้นจริง เส้นยิ่งใกล้เส้นทแยงมุมยิ่งแปลค่า Probability ตรงไปตรงมาได้ดี

In [ ]:
fraction_positive, mean_probability = calibration_curve(
    test_predictions[TARGET_COLUMN].astype(int),
    test_predictions["predicted_probability"],
    n_bins=10,
    strategy="quantile",
)

plt.figure(figsize=(6, 5))
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Perfect calibration")
plt.plot(mean_probability, fraction_positive, marker="o", label="CatBoost")
plt.xlabel("Mean predicted probability")
plt.ylabel("Observed positive rate")
plt.title("Test Calibration Curve")
plt.legend()
plt.tight_layout()
plt.show()

print(f"Test Log Loss: {test_metrics['log_loss']:.3f}")
print(f"Test Brier Score: {test_metrics['brier_score']:.3f}")

## 11. Threshold Analysis จาก Validation

Threshold ไม่ใช่พารามิเตอร์ฝึกโมเดล แต่เป็นจุดตัดสำหรับเปลี่ยน Probability เป็นคำตอบ True/False การเลือกต้องทำจาก Validation เท่านั้น และควรพิจารณาต้นทุนของ False Positive กับ False Negative

In [ ]:
best_threshold_row = threshold_analysis.loc[
    threshold_analysis["balanced_accuracy"].idxmax()
]

plt.figure(figsize=(9, 5))
for metric in ["balanced_accuracy", "precision", "recall", "specificity"]:
    plt.plot(
        threshold_analysis["threshold"],
        threshold_analysis[metric],
        marker="o",
        label=metric.replace("_", " "),
    )
plt.axvline(0.50, color="gray", linestyle="--", label="Current threshold")
plt.axvline(
    best_threshold_row["threshold"], color="red", linestyle=":",
    label="Best balanced accuracy",
)
plt.xlabel("Probability threshold")
plt.ylabel("Metric value")
plt.title("Validation Threshold Trade-off")
plt.ylim(0, 1.02)
plt.legend(ncol=2)
plt.tight_layout()
plt.show()

best_threshold_row.to_frame("Validation value").style.format("{:.3f}")

Threshold ที่ทำให้ Balanced Accuracy สูงสุดเป็นเพียงตัวเลือกทางสถิติ หากธุรกิจต้องการลดห้องว่างและยอมติดตามประกาศเพิ่ม ควรให้ความสำคัญกับ Recall แต่หากทีมมีทรัพยากรจำกัดควรเพิ่ม Precision

## 12. CatBoost Feature Importance

Feature Importance แสดงว่าตัวแปรใดถูกใช้ลดความผิดพลาดของโมเดลมาก แต่ไม่ได้แปลว่าตัวแปรนั้นเป็นสาเหตุของการปล่อยเช่า

In [ ]:
top_features = feature_importance.head(10).sort_values("importance")

plt.figure(figsize=(9, 5))
sns.barplot(data=top_features, x="importance", y="feature", color="#457b9d")
plt.xlabel("CatBoost importance")
plt.ylabel("Feature")
plt.title("Top 10 Features")
plt.tight_layout()
plt.show()

feature_importance.head(10)

## 13. Error Analysis แยกตามโครงการ

การดูคะแนนรวมอย่างเดียวอาจซ่อนโครงการที่ Probability สูงหรือต่ำเกินจริง จึงเปรียบเทียบค่าเฉลี่ยที่ทำนายกับอัตราที่เกิดจริงในแต่ละโครงการ

In [ ]:
test_predictions["false_positive"] = (
    ~test_predictions[TARGET_COLUMN].astype(bool)
    & test_predictions["predicted_class"].astype(bool)
)
test_predictions["false_negative"] = (
    test_predictions[TARGET_COLUMN].astype(bool)
    & ~test_predictions["predicted_class"].astype(bool)
)
project_errors = (
    test_predictions.groupby("project_id", as_index=False)
    .agg(
        listings=(TARGET_COLUMN, "size"),
        actual_rate=(TARGET_COLUMN, "mean"),
        predicted_rate=("predicted_probability", "mean"),
        false_positive=("false_positive", "sum"),
        false_negative=("false_negative", "sum"),
    )
)
project_errors["calibration_gap"] = (
    project_errors["predicted_rate"] - project_errors["actual_rate"]
)
project_errors["absolute_gap"] = project_errors["calibration_gap"].abs()
project_errors = project_errors.sort_values("absolute_gap", ascending=False)
project_errors.head(10).style.format({
    "actual_rate": "{:.1%}",
    "predicted_rate": "{:.1%}",
    "calibration_gap": "{:+.1%}",
    "absolute_gap": "{:.1%}",
})

In [ ]:
plot_errors = project_errors.head(10).sort_values("calibration_gap")

plt.figure(figsize=(9, 5))
colors = np.where(plot_errors["calibration_gap"] >= 0, "#e76f51", "#2a9d8f")
plt.barh(plot_errors["project_id"].astype(str), plot_errors["calibration_gap"], color=colors)
plt.axvline(0, color="gray", linewidth=1)
plt.xlabel("Predicted rate - actual rate")
plt.ylabel("Project ID")
plt.title("Largest Project-level Calibration Gaps on Test")
plt.tight_layout()
plt.show()

## 14. Feature Ablation

Ablation คือการตัด Feature บางกลุ่มออกแล้วฝึกใหม่ เพื่อทดสอบว่าโมเดลพึ่งข้อมูลกลุ่มนั้นมากเพียงใด การทดลองนี้ใช้ Train และ Validation เท่านั้น

In [ ]:
ablation_plot = feature_ablation.sort_values("roc_auc")
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.barplot(data=ablation_plot, x="roc_auc", y="feature_set", color="#457b9d", ax=axes[0])
sns.barplot(data=ablation_plot, x="log_loss", y="feature_set", color="#e9c46a", ax=axes[1])
axes[0].set_title("Validation ROC AUC: Higher is Better")
axes[1].set_title("Validation Log Loss: Lower is Better")
axes[0].set_ylabel("Feature set")
axes[1].set_ylabel("")
plt.tight_layout()
plt.show()

feature_ablation.style.format({
    "roc_auc": "{:.3f}",
    "average_precision": "{:.3f}",
    "log_loss": "{:.3f}",
    "brier_score": "{:.3f}",
    "balanced_accuracy": "{:.3f}",
})

ผล Ablation แสดงว่า Feature เวลาและฤดูกาลเป็นข้อมูลจำเป็น การตัดออกทำให้ Validation AUC ลดจากประมาณ 0.939 เหลือ 0.797–0.871 จึงคง Feature ชุดเดิมไว้

## 15. Rolling Time Backtest ภายใน Train

แต่ละรอบฝึกจากอดีตทั้งหมดก่อนต้นไตรมาส แล้วตรวจสอบกับไตรมาสถัดไป วิธีนี้ช่วยวัดความเสถียรโดยไม่เปิดดู Test

In [ ]:
plt.figure(figsize=(9, 5))
for metric in ["roc_auc", "balanced_accuracy"]:
    plt.plot(
        rolling_metrics["fold"], rolling_metrics[metric],
        marker="o", linewidth=2, label=metric.replace("_", " "),
    )
plt.ylim(0.5, 1.02)
plt.xlabel("Evaluation quarter")
plt.ylabel("Metric value")
plt.title("CatBoost Rolling Time Backtest")
plt.legend()
plt.tight_layout()
plt.show()

rolling_metrics.style.format({
    "check_positive_rate": "{:.1%}",
    "roc_auc": "{:.3f}",
    "average_precision": "{:.3f}",
    "log_loss": "{:.3f}",
    "brier_score": "{:.3f}",
    "balanced_accuracy": "{:.3f}",
})

ผล Rolling Backtest มี ROC AUC ระหว่างประมาณ 0.874–0.986 ทุกไตรมาสสูงกว่า 0.5 ชัดเจน ไตรมาส 2/2025 เป็นช่วงที่ยากที่สุด จึงควรติดตามผลแยกตามฤดูกาลต่อไป

## 16. ข้อสรุปสำหรับรายงาน

- CatBoost สามารถจัดอันดับโอกาสปล่อยเช่าได้ค่อนข้างดี แต่ผล Test ต่ำกว่า Validation แสดงถึง temporal distribution shift
- ควรรายงาน Probability metrics เช่น Log Loss และ Brier Score ควบคู่กับ AUC และ F1
- Threshold 0.50 เป็นค่าเริ่มต้น ไม่ใช่ค่าที่ดีที่สุดสำหรับทุกเป้าหมายธุรกิจ
- Feature Importance ใช้อธิบายพฤติกรรมโมเดล ไม่สามารถสรุปเหตุและผล
- Ablation ยืนยันว่ากลุ่ม Feature เวลาและฤดูกาลมีข้อมูลที่จำเป็นต่อการพยากรณ์
- Rolling Backtest แสดงว่าโมเดลแยกคลาสได้ดีทุกไตรมาสในปี 2025 แต่ประสิทธิภาพเปลี่ยนตามฤดูกาล
- ก่อนใช้งานจริงควรติดตาม Calibration และข้อผิดพลาดแยกตามโครงการเป็นระยะ